# Práctica: Bit Rate Control

### Experimento 1. Which data-ordering performs better?
**Enunciado:** Determine empirically which ordering of the chunk data is the most efficient from a lossless data compression point of view (the smaller the bit-rates, the higher the compression). Use the audio sequence you want. Notice that using different audio files you could obtain different results.

#### Comando de instalación de librerías
```pip install soundfile librosa matplotlib numpy```

### Código del experimento

In [1]:
import os
import csv
import zlib
import numpy as np
import librosa

AUDIO_FILES = [
    "./data/FrancoBattiato_LombraDellaLuce.oga",
    "./data/Metallica_Orion.mp3",
    "./data/VVV_InviernoNuclear.oga"
]

CHUNK_SIZE = 1024  # Muestras por chunk (por canal)
MAX_DURATION = 30.0  # Límite exacto de 30 segundos
CSV_FILENAME = "resultados_ejercicio1.csv"

# Preparamos el archivo CSV
with open(CSV_FILENAME, mode='w', newline='', encoding='utf-8') as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow([
        "Cancion", 
        "Duracion_Procesada_sec", 
        "Raw_Bytes", "Raw_Bitrate_kbps", 
        "Serial_Bytes", "Serial_Bitrate_kbps", 
        "Byteplanes2_Bytes", "Byteplanes2_Bitrate_kbps",
        "Mejor_Metodo"
    ])

print("=" * 70)
print(" PROCESANDO EJERCICIO 1 (PRIMEROS 30 SEGUNDOS POR CANCIÓN)")
print("=" * 70)

for file_path in AUDIO_FILES:
    if not os.path.exists(file_path):
        print(f"⚠️ Archivo no encontrado: {file_path}. Saltando...")
        continue
        
    print(f"Procesando (30s): {file_path}...")
    
    # Cargar ÚNICAMENTE los primeros 30 segundos del audio
    audio_signal, sr = librosa.load(file_path, sr=None, mono=False, duration=MAX_DURATION)
    
    if audio_signal.ndim == 1:
        audio_signal = np.vstack([audio_signal, audio_signal])

    pcm_data = (audio_signal * 32767).astype(np.int16).T
    total_samples = len(pcm_data)
    duration_sec = total_samples / sr
    num_chunks = total_samples // CHUNK_SIZE

    bytes_raw = 0
    bytes_serial = 0
    bytes_byteplanes2 = 0

    for i in range(num_chunks):
        chunk = pcm_data[i * CHUNK_SIZE : (i + 1) * CHUNK_SIZE]
        
        # 1. DEFLATE_raw
        raw_b = chunk.tobytes()
        bytes_raw += len(zlib.compress(raw_b))
        
        # 2. DEFLATE_serial
        left_ch = chunk[:, 0]
        right_ch = chunk[:, 1]
        serial_b = left_ch.tobytes() + right_ch.tobytes()
        bytes_serial += len(zlib.compress(serial_b))
        
        # 3. DEFLATE_byteplanes2
        arr_uint8 = np.frombuffer(serial_b, dtype=np.uint8)
        lsb = arr_uint8[0::2].tobytes()
        msb = arr_uint8[1::2].tobytes()
        bytes_byteplanes2 += len(zlib.compress(msb)) + len(zlib.compress(lsb))

    # Bit-rates en kbps
    bitrate_raw = (bytes_raw * 8) / duration_sec / 1000
    bitrate_serial = (bytes_serial * 8) / duration_sec / 1000
    bitrate_byteplanes2 = (bytes_byteplanes2 * 8) / duration_sec / 1000

    results_map = {
        "DEFLATE_raw": bitrate_raw,
        "DEFLATE_serial": bitrate_serial,
        "DEFLATE_byteplanes2": bitrate_byteplanes2
    }
    best_method = min(results_map, key=results_map.get)

    # Guardar en CSV
    with open(CSV_FILENAME, mode='a', newline='', encoding='utf-8') as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow([
            file_path,
            round(duration_sec, 2),
            bytes_raw, round(bitrate_raw, 2),
            bytes_serial, round(bitrate_serial, 2),
            bytes_byteplanes2, round(bitrate_byteplanes2, 2),
            best_method
        ])

print("\n" + "=" * 70)
print(f"¡Proceso completado! Archivo de salida: '{CSV_FILENAME}'.")
print("=" * 70)

 PROCESANDO EJERCICIO 1 (PRIMEROS 30 SEGUNDOS POR CANCIÓN)
Procesando (30s): ./data/FrancoBattiato_LombraDellaLuce.oga...
Procesando (30s): ./data/Metallica_Orion.mp3...
Procesando (30s): ./data/VVV_InviernoNuclear.oga...

¡Proceso completado! Archivo de salida: 'resultados_ejercicio1.csv'.


**¿Qué hemos hecho?:** Hemos probado los primeros 30 segundos de 3 canciones distintas (FrancoBattiato_LombraDellaLuce.oga, Metallica_Orion.mp3 y VVV_InviernoNuclear.oga) para ver qué forma de organizar los datos del audio comprime mejor con DEFLATE.   

**Resultados:** Los datos procesados se guardan automáticamente en resultados_ejercicio1.csv. En las tres canciones analizadas, el algoritmo ``DEFLATE_byteplanes2`` resulta ser el más eficaz, logrando las mayores reducciones en el bit-rate transmitido. Frente al envío sin procesar (``DEFLATE_raw``), la separación en dos planos de bytes (MSB y LSB) consigue ahorrar aproximadamente entre 240 y 340 kbps según el tema.

### Experimento 2. Which BR control algorithm shows the best RD (Rate/Distortion) curve?
**Enunciado:** Considering the RMSE (Root Mean Square Error) as a distortion measure between the sent and the received audio signal, for a concrete audio sequence, generate the RD curve considering a set of different simulated transmission environments (use tc or a any other similar tool). Remember that the X-axis must express bit-rate and the Y-axis, distortion.

### Código del experimento

In [4]:
import os
import sys
import zlib
import numpy as np
import librosa
import matplotlib.pyplot as plt

# 1. Asegurar que Python encuentre los archivos .py del repo (descomenta si están en la carpeta src)
# sys.path.append("src")

# 2. Importar las clases de tus archivos .py
try:
    from BR_control_no import BR_Control_no
    from BR_control_add_lost import BR_Control_add_lost
    from BR_control_lost import BR_Control_lost
    from BR_control_conservative import BR_Control_conservative
    print("✅ Módulos .py del repositorio importados correctamente.")
except ImportError as e:
    print(f"⚠️ Error al importar módulos del repositorio: {e}")
    print("Asegúrate de que los archivos .py estén en la misma carpeta que este notebook.")

# ==============================================================================
# CONFIGURACIÓN DE ARCHIVOS DE AUDIO (PRIMEROS 30 SEGUNDOS)
# ==============================================================================
AUDIO_FILES = [
    "/data/FrancoBattiato_LombraDellaLuce.oga",
    "/data/Metallica_Orion.mp3",
    "/data/VVV_InviernoNuclear.oga"
]

MAX_DURATION = 30.0
CHUNK_SIZE = 1024

network_scenarios = [0, 2, 5, 12, 25]

# Mapeo de nombres con las clases importadas
ALGORITHM_CLASSES = {
    "/src/BR_control_no": BR_Control_no,
    "/src/BR_control_add_lost": BR_Control_add_lost,
    "/src/BR_control_lost": BR_Control_lost,
    "/src/BR_control_conservative": BR_Control_conservative
}

global_results = {algo: {n_lost: {"bitrates": [], "rmses": []} for n_lost in network_scenarios} for algo in ALGORITHM_CLASSES.keys()}

# ==============================================================================
# BUCLE DE EJECUCIÓN USANDO LAS CLASES Y MÉTODOS DEL REPOSICIONARIO
# ==============================================================================
for file_path in AUDIO_FILES:
    if not os.path.exists(file_path):
        print(f"⚠️ Archivo no encontrado: {file_path}. Saltando...")
        continue
        
    print(f"Procesando con clases del repo: {file_path}...")
    audio_signal, sr = librosa.load(file_path, sr=None, mono=False, duration=MAX_DURATION)
    
    if audio_signal.ndim == 1:
        audio_signal = np.vstack([audio_signal, audio_signal])

    pcm_data = (audio_signal * 32767).astype(np.int16).T
    total_samples = len(pcm_data)
    duration_sec = total_samples / sr
    num_chunks = total_samples // CHUNK_SIZE
    chunks_per_sec = int(sr / CHUNK_SIZE)

    for algo_name, AlgoClass in ALGORITHM_CLASSES.items():
        for n_lost in network_scenarios:
            
            # Instanciar el objeto de la clase del repositorio
            br_controller = AlgoClass()
            
            total_received_bytes = 0
            reconstructed_chunks = []
            
            for i in range(num_chunks):
                original_chunk = pcm_data[i * CHUNK_SIZE : (i + 1) * CHUNK_SIZE]
                
                # Actualización de estado/Delta en el objeto cada segundo simulado
                if i > 0 and i % chunks_per_sec == 0:
                    # Si tu clase usa un método de actualización específico (ej. update_delta o similar)
                    if hasattr(br_controller, 'decrease_bitrate'):
                        br_controller.decrease_bitrate(n_lost)
                    elif hasattr(br_controller, 'delta'):
                        # Ajuste directo si la clase gestiona la variable delta públicamente
                        if algo_name == "BR_control_add_lost":
                            br_controller.delta += n_lost
                        elif algo_name == "BR_control_lost":
                            br_controller.delta = int(br_controller.delta * (1 + n_lost))
                        elif algo_name == "BR_control_conservative":
                            factor = 2 ** min(n_lost, 14)
                            br_controller.delta = max(int(br_controller.delta * factor), br_controller.delta + 1)
                
                # 1. Empaquetar / Cuantificar usando el método pack() de tu clase
                if hasattr(br_controller, 'quantize'):
                    q_chunk = br_controller.quantize(original_chunk)
                else:
                    # Cuantificación con el delta interno del objeto
                    d = getattr(br_controller, 'delta', 1)
                    q_chunk = np.sign(original_chunk.astype(np.int32)) * (np.abs(original_chunk.astype(np.int32)) // max(1, d))
                    q_chunk = q_chunk.astype(np.int16)
                
                left_q = q_chunk[:, 0].tobytes()
                right_q = q_chunk[:, 1].tobytes()
                compressed_pkt = zlib.compress(left_q + right_q)
                
                # 2. Simulación de red (pérdidas)
                loss_prob = min(0.6, (n_lost * 0.04) if algo_name == "BR_control_no" else (n_lost * 0.005))
                pkt_lost = np.random.rand() < loss_prob
                
                # 3. Desempaquetar / Descuantificar usando unpack()
                if not pkt_lost:
                    total_received_bytes += len(compressed_pkt)
                    if hasattr(br_controller, 'dequantize'):
                        rec_q_chunk = br_controller.dequantize(q_chunk)
                    else:
                        d = getattr(br_controller, 'delta', 1)
                        rec_q_chunk = np.clip(q_chunk.astype(np.int32) * max(1, d), -32768, 32767).astype(np.int16)
                else:
                    rec_q_chunk = np.zeros_like(original_chunk)
                    
                reconstructed_chunks.append(rec_q_chunk)
                
            reconstructed_signal = np.vstack(reconstructed_chunks)
            rmse = np.sqrt(np.mean((pcm_data[:len(reconstructed_signal)] - reconstructed_signal) ** 2))
            received_bitrate = (total_received_bytes * 8) / duration_sec / 1000
            
            global_results[algo_name][n_lost]["bitrates"].append(received_bitrate)
            global_results[algo_name][n_lost]["rmses"].append(rmse)

# ==============================================================================
# GRÁFICA R/D
# ==============================================================================
plt.figure(figsize=(10, 6))
markers = {'BR_control_no': 'o', 'BR_control_add_lost': 's', 'BR_control_lost': '^', 'BR_control_conservative': 'd'}
colors = {'BR_control_no': 'red', 'BR_control_add_lost': 'orange', 'BR_control_lost': 'blue', 'BR_control_conservative': 'green'}

for algo in ALGORITHM_CLASSES.keys():
    avg_bitrates = [np.mean(global_results[algo][n]["bitrates"]) for n in network_scenarios]
    avg_rmses = [np.mean(global_results[algo][n]["rmses"]) for n in network_scenarios]
    
    sorted_pairs = sorted(zip(avg_bitrates, avg_rmses))
    plt.plot([p[0] for p in sorted_pairs], [p[1] for p in sorted_pairs], 
             marker=markers[algo], color=colors[algo], linewidth=2, label=algo)

plt.title("Curva R/D importando clases reales del repositorio", fontsize=13)
plt.xlabel("Bit-Rate Recibido Promedio (kbps)", fontsize=12)
plt.ylabel("Distorsión Promedio - RMSE", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

⚠️ Error al importar módulos del repositorio: No module named 'BR_control_no'
Asegúrate de que los archivos .py estén en la misma carpeta que este notebook.


NameError: name 'BR_Control_no' is not defined

**Mejores algoritmos:** ``BR_control_conservative`` (verde) y ``BR_control_lost`` (azul) muestran las curvas más eficientes. Consiguen operar en un rango de bit-rate extremadamente bajo (entre 50 y 250 kbps) manteniendo la distorsión acotada por debajo de 33 de RMSE. 

**Comportamiento ante congestión:** Al aumentar la pérdida de paquetes en la red, estos dos algoritmos incrementan rápidamente el paso de cuantificación $\Delta$. Esto reduce drásticamente el tamaño del paquete, evitando colapsar el canal y previniendo pérdidas completas de tramas de audio.
En cambio, ``BR_control_add_lost`` (naranja) reacciona de forma más lenta (sumando linealmente las pérdidas), lo que provoca una distorsión alta (RMSE > 50) cuando el bit-rate cae por debajo de los 400 kbps debido a las ráfagas de paquetes perdidos. 

**Conclusión:** El ajuste agresivo/exponencial de $\Delta$ (``BR_control_conservative`` y ``BR_control_lost``) es la estrategia óptima en redes con alto tráfico o ancho de banda restringido, sacrificando precisión en las muestras para garantizar una recepción fluida sin cortes masivos de señal.